# 04 — Cross-Variant Service Stability

For each (language × service), compare translations across all four prompt variants:

> *Does the same LLM give the same answer regardless of how it was asked?*

- **Baseline services** (Wikipedia, GT, EasyNMT, Lingvanex) are prompt-invariant by design — they should always score 1.0 and serve as a sanity check.
- **LLM services** (OpenAI, Claude, Gemini, Ollama) are the focus: instability here means the model has no confident grounded answer, and framing shifts the output.
- **Judge** variant is included for completeness but is a synthesis step — its stability reflects how well it synthesises the other three variants, not independent prompt sensitivity.

Key metric: **agreement_rate** = fraction of variants that produced the same best translation for a given (language × service).

Input: `translated_terms/digital_humanities/evaluation/across_variant_detail.csv`  
       `translated_terms/digital_humanities/evaluation/across_variant_service_summary.csv`

In [1]:
import os
import sys
import pandas as pd
import altair as alt
from pathlib import Path

alt.data_transformers.enable('vegafusion')

sys.path.insert(0, str(Path('..').resolve()))
from scripts.utils import get_data_directory_path, read_csv_file, get_language_family
from scripts.exploration.explore_confidence_across_variants import (
    run_across_variant_evaluation, LLM_SERVICES, BASELINE_SERVICES
)

DATA_DIR  = get_data_directory_path()
TERM      = 'Digital Humanities'
TERM_SLUG = TERM.lower().replace(' ', '_')
EVAL_DIR  = os.path.join(DATA_DIR, 'translated_terms', TERM_SLUG, 'evaluation')

SERVICE_ORDER = [
    'Wikipedia', 'Google Translate', 'EasyNMT', 'Lingvanex',
    'OpenAI', 'Claude', 'Gemini', 'Ollama',
]
SERVICE_COLOURS = {
    'Wikipedia':       '#aec7e8',
    'Google Translate':'#c5b0d5',
    'EasyNMT':         '#c49c94',
    'Lingvanex':       '#dbdb8d',
    'OpenAI':          '#1f77b4',
    'Claude':          '#ff7f0e',
    'Gemini':          '#2ca02c',
    'Ollama':          '#d62728',
}

## 3.1 Load Data

In [2]:
detail_path  = os.path.join(EVAL_DIR, 'across_variant_detail.csv')
summary_path = os.path.join(EVAL_DIR, 'across_variant_service_summary.csv')

if not os.path.exists(detail_path):
    print('Running explore_confidence_across_variants.py...')
    detail_df, summary_df = run_across_variant_evaluation(DATA_DIR, [TERM])
else:
    detail_df  = read_csv_file(detail_path)
    summary_df = read_csv_file(summary_path)

# Only rows where a service actually produced data
has_data = detail_df[detail_df['n_variants_present'] > 0].copy()
llm_df   = has_data[~has_data['is_baseline']].copy()

print(f'detail_df:  {len(detail_df)} rows ({detail_df["language_code"].nunique()} languages × {detail_df["service"].nunique()} services)')
print(f'has_data:   {len(has_data)} rows with translations')
print(f'LLM rows:   {len(llm_df)}')
summary_df

detail_df:  6864 rows (857 languages × 8 services)
has_data:   3864 rows with translations
LLM rows:   3373


,service,n_languages,mean_agreement,std_agreement,is_baseline,cv_agreement
0,EasyNMT,100,1.0000,0.0000,True,0.0000
1,Google Translate,242,1.0000,0.0000,True,0.0000
2,Lingvanex,109,1.0000,0.0000,True,0.0000
3,Wikipedia,40,1.0000,0.0000,True,0.0000
4,Claude,857,0.5928,0.2678,False,0.4518
5,Gemini,855,0.5864,0.2554,False,0.4355
6,OpenAI,857,0.4657,0.2359,False,0.5065
7,Ollama,804,0.3556,0.1887,False,0.5307


## 3.2 Service Stability Overview

Mean agreement rate per service across all languages. Baselines should all score 1.0.

In [3]:
summary_plot = summary_df.copy()
summary_plot['is_baseline'] = summary_plot['is_baseline'].astype(bool)
summary_plot['service_type'] = summary_plot['is_baseline'].map({True: 'Baseline', False: 'LLM'})

bars = alt.Chart(summary_plot).mark_bar().encode(
    x=alt.X('mean_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean agreement rate across variants'),
    y=alt.Y('service:N', sort=SERVICE_ORDER, title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()), range=list(SERVICE_COLOURS.values())),
        legend=None),
    opacity=alt.condition(
        alt.datum.is_baseline,
        alt.value(0.4),
        alt.value(1.0)
    ),
    tooltip=['service:N', 'mean_agreement:Q', 'std_agreement:Q', 'cv_agreement:Q',
             'n_languages:Q', 'service_type:N']
)

text = alt.Chart(summary_plot).mark_text(align='left', dx=4, fontSize=11).encode(
    x=alt.X('mean_agreement:Q'),
    y=alt.Y('service:N', sort=SERVICE_ORDER),
    text=alt.Text('mean_agreement:Q', format='.2f')
)

(bars + text).properties(
    width=450, height=280,
    title='Service Stability — Mean Cross-Variant Agreement Rate'
)

alt.LayerChart(...)

## 3.3 Agreement Rate Distribution per LLM Service

Boxplot of per-language agreement rates. A wide spread means the service is stable for some languages but volatile for others — which is itself informative about where the model has grounded knowledge.

In [4]:
alt.Chart(llm_df).mark_boxplot(extent='min-max', size=20).encode(
    x=alt.X('agreement_rate:Q', scale=alt.Scale(domain=[0, 1]),
            title='Agreement rate (per language)'),
    y=alt.Y('service:N',
            sort=[s for s in SERVICE_ORDER if s in LLM_SERVICES],
            title=None),
    color=alt.Color('service:N',
        scale=alt.Scale(domain=list(SERVICE_COLOURS.keys()), range=list(SERVICE_COLOURS.values())),
        legend=None)
).properties(
    width=450, height=200,
    title='Per-Language Agreement Rate Distribution (LLM services)'
)

alt.Chart(...)

## 3.4 Stability by Language Family

Which language families show the most prompt-driven variation? High variance here suggests the LLM's knowledge of those languages is thin enough that prompt framing shifts the output.

In [5]:
family_service = (
    llm_df
    .groupby(['language_family', 'service'])['agreement_rate']
    .mean()
    .reset_index()
    .rename(columns={'agreement_rate': 'mean_agreement'})
)

# Sort families by overall mean agreement (most stable first)
family_order = (
    family_service.groupby('language_family')['mean_agreement']
    .mean().sort_values(ascending=False).index.tolist()
)

alt.Chart(family_service).mark_rect().encode(
    x=alt.X('service:N',
            sort=[s for s in SERVICE_ORDER if s in LLM_SERVICES],
            title=None),
    y=alt.Y('language_family:N', sort=family_order, title='Language Family'),
    color=alt.Color('mean_agreement:Q',
        scale=alt.Scale(scheme='blues', domain=[0, 1]),
        title='Mean agreement'),
    tooltip=['language_family:N', 'service:N',
             alt.Tooltip('mean_agreement:Q', format='.2f')]
).properties(
    width=350, height=420,
    title='Mean Cross-Variant Agreement by Language Family × LLM Service'
)

alt.Chart(...)

In [ ]:
family_counts = (
    llm_df.groupby('language_family')['language_code']
    .nunique()
    .reset_index(name='n_languages')
)

family_service_line = family_service.merge(family_counts, on='language_family')
family_service_line['family_label'] = (
    family_service_line['language_family']
    + ' (n=' + family_service_line['n_languages'].astype(str) + ')'
)

llm_service_order = [s for s in SERVICE_ORDER if s in LLM_SERVICES]

alt.Chart(family_service_line).mark_line(point=True).encode(
    x=alt.X('service:N', sort=llm_service_order, title='LLM service'),
    y=alt.Y('mean_agreement:Q', scale=alt.Scale(domain=[0, 1]), title='Mean agreement rate'),
    color=alt.Color('family_label:N', title='Language family'),
    tooltip=[
        'family_label:N',
        'service:N',
        alt.Tooltip('mean_agreement:Q', format='.3f'),
    ],
).properties(
    width=500, height=380,
    title='Mean Cross-Variant Agreement per LLM Service by Language Family',
)

## 3.5 Prompt Variant Pairwise Similarity

For each pair of prompt variants, what fraction of (language × LLM service) combinations produced the same translation? This shows which variants are effectively interchangeable vs. which pull the model in different directions.

In [6]:
import ast

VARIANTS = ['minimal', 'expert_persona', 'native_rationale', 'judge']

# Build variant→translation lookup from variant_translations column
pair_matches = {(a, b): [] for a in VARIANTS for b in VARIANTS if a < b}

for _, row in llm_df.iterrows():
    try:
        vt = ast.literal_eval(str(row['variant_translations']))
    except Exception:
        continue
    for (a, b) in pair_matches:
        ta = vt.get(a)
        tb = vt.get(b)
        if ta and tb:
            pair_matches[(a, b)].append(
                1 if str(ta).strip().lower() == str(tb).strip().lower() else 0
            )

rows = []
for (a, b), matches in pair_matches.items():
    if matches:
        sim = sum(matches) / len(matches)
        rows.append({'variant_a': a, 'variant_b': b, 'similarity': round(sim, 3)})
        rows.append({'variant_a': b, 'variant_b': a, 'similarity': round(sim, 3)})
# Diagonal
for v in VARIANTS:
    rows.append({'variant_a': v, 'variant_b': v, 'similarity': 1.0})

pair_df = pd.DataFrame(rows)

VARIANT_LABELS = {
    'minimal':          'Minimal',
    'expert_persona':   'Expert Persona',
    'native_rationale': 'Native Rationale',
    'judge':            'Judge',
}
pair_df['label_a'] = pair_df['variant_a'].map(VARIANT_LABELS)
pair_df['label_b'] = pair_df['variant_b'].map(VARIANT_LABELS)
label_order = list(VARIANT_LABELS.values())

heatmap = alt.Chart(pair_df).mark_rect().encode(
    x=alt.X('label_a:N', sort=label_order, title=None),
    y=alt.Y('label_b:N', sort=label_order, title=None),
    color=alt.Color('similarity:Q',
        scale=alt.Scale(scheme='greens', domain=[0, 1]),
        title='Fraction same'),
    tooltip=['label_a:N', 'label_b:N',
             alt.Tooltip('similarity:Q', format='.2f')]
)

text_layer = alt.Chart(pair_df).mark_text(fontSize=11).encode(
    x=alt.X('label_a:N', sort=label_order),
    y=alt.Y('label_b:N', sort=label_order),
    text=alt.Text('similarity:Q', format='.2f'),
    color=alt.condition(
        alt.datum.similarity > 0.6,
        alt.value('white'),
        alt.value('black')
    )
)

(heatmap + text_layer).properties(
    width=350, height=300,
    title='Prompt Variant Pairwise Similarity (LLM services only)'
)

alt.LayerChart(...)

## 3.6 Most Divergent Languages

Languages where LLM services diverge most across prompt variants — the lowest mean agreement rates. These are where prompt framing matters most, and where the model's knowledge is thinnest.

In [7]:
# Mean LLM agreement per language
lang_stability = (
    llm_df
    .groupby(['language_code', 'language_name', 'language_family'])['agreement_rate']
    .mean()
    .reset_index()
    .rename(columns={'agreement_rate': 'mean_llm_agreement'})
    .sort_values('mean_llm_agreement')
)

bottom30 = lang_stability.head(30).copy()

alt.Chart(bottom30).mark_bar().encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean LLM agreement rate'),
    y=alt.Y('language_name:N', sort='-x', title=None),
    color=alt.Color('language_family:N', title='Family'),
    tooltip=['language_code:N', 'language_name:N', 'language_family:N',
             alt.Tooltip('mean_llm_agreement:Q', format='.2f')]
).properties(
    width=400, height=500,
    title='30 Least Stable Languages (mean LLM cross-variant agreement)'
)

alt.Chart(...)

## 3.7 Agreement Rate vs. Wikipedia Coverage

Languages with Wikipedia DH translations should show higher LLM stability — the model has community-grounded text to draw on. This chart tests that hypothesis.

In [8]:
wiki_df = detail_df[detail_df['service'] == 'Wikipedia'][['language_code', 'term_source', 'best_candidate']].copy()
wiki_df['has_wikipedia'] = wiki_df['best_candidate'].notna() & (wiki_df['best_candidate'].astype(str).str.strip() != 'nan')

lang_wiki = lang_stability.merge(
    wiki_df[['language_code', 'has_wikipedia']],
    on='language_code', how='left'
)
lang_wiki['has_wikipedia'] = lang_wiki['has_wikipedia'].fillna(False)
lang_wiki['wikipedia_label'] = lang_wiki['has_wikipedia'].map({True: 'Wikipedia translation exists', False: 'No Wikipedia translation'})

base = alt.Chart(lang_wiki)

scatter = base.mark_circle(opacity=0.6, size=60).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1]),
            title='Mean LLM cross-variant agreement'),
    color=alt.Color('wikipedia_label:N',
        scale=alt.Scale(
            domain=['Wikipedia translation exists', 'No Wikipedia translation'],
            range=['#2ca02c', '#d62728']
        ), title=None),
    tooltip=['language_name:N', 'language_family:N',
             alt.Tooltip('mean_llm_agreement:Q', format='.2f'), 'wikipedia_label:N']
).properties(width=400, height=250)

strip = base.mark_tick(thickness=2, bandSize=12).encode(
    x=alt.X('mean_llm_agreement:Q', scale=alt.Scale(domain=[0, 1])),
    y=alt.Y('wikipedia_label:N', title=None),
    color=alt.Color('wikipedia_label:N',
        scale=alt.Scale(
            domain=['Wikipedia translation exists', 'No Wikipedia translation'],
            range=['#2ca02c', '#d62728']
        ), legend=None),
    tooltip=['language_name:N', alt.Tooltip('mean_llm_agreement:Q', format='.2f')]
).properties(width=400, height=100,
             title='LLM Stability vs Wikipedia Coverage')

# Summary stats
print(lang_wiki.groupby('wikipedia_label')['mean_llm_agreement'].agg(['mean','median','count']).round(3))

alt.vconcat(scatter, strip).resolve_scale(color='shared')

                               mean  median  count
wikipedia_label                                   
No Wikipedia translation      0.491   0.458    817
Wikipedia translation exists  0.763   0.802     40


alt.VConcatChart(...)